In [ ]:
import torch
import pickle
import os
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
from pylab import rcParams
import matplotlib.pyplot as plt
from matplotlib import rc
from sklearn.preprocessing import MinMaxScaler
from pandas.plotting import register_matplotlib_converters
from torch import nn, optim
%matplotlib inline
%config InlineBackend.figure_format='retina'

sns.set(style='whitegrid', palette='muted', font_scale=1.2)
rcParams['figure.figsize'] = 14, 10
register_matplotlib_converters()
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DATA Visualization
df = pd.read_excel('APY_Rice.xls')
df.head
df.info()
df.columns
sum_maxp = df["Production (Tonnes)"].sum()
df["percent_of_production"] = df["Production (Tonnes)"].map(lambda x:(x/sum_maxp)*100)
df.shape
df.dtypes
df.isnull().sum()
df.dropna(inplace=True)
df.isna().sum()
cols = ['Area (Hectare)', 'Yield (Tonnes/Hectare)', 'percent_of_production', 'Production (Tonnes)']
Data Pre-Processing
for temp in cols:
  Q1 = df[temp].quantile(0.25)
  Q3 = df[temp].quantile(0.75)
  IQR = Q3 - Q1

  # identify outliers
  threshold = 1.5
  outliers = df[(df[temp] < Q1 - threshold * IQR) | (df[temp] > Q3 + threshold * IQR)]
  df.drop(outliers.index,inplace=True)
  plt.figure(figsize=(7,7))
for j , col in enumerate(cols):
    plt.subplot(3, 3, j + 1)
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot for {col}')

plt.tight_layout()
plt.show()
for i in df.columns:
    print(i,"  ",df[i].unique())
df['Season'].unique()
for state in df['State'].unique():
    # Extract unique districts for the current state
    districts = df[df['State'] == state]['District'].unique()
    
    # Concatenate districts into a single string
    district_str = ", ".join(districts)
    
    # Print the state and its districts
    print(f"State: {state}")
    print(f"Districts: {district_str}")
    print()
state_budget=df.groupby('State')['Production (Tonnes)'].sum()
state_budget=state_budget.sort_values(ascending=False)
plt.xlabel('Production')
sns.barplot(x=state_budget.values,y=state_budget.index)
plt.figure(figsize=(18,10))
sns.lineplot(x=df['Year'],y=df['Production (Tonnes)'])
plt.xticks(rotation=90)
plt.figure(figsize=(10,9))
season_budget=df.groupby('Season')['Production (Tonnes)'].sum()
season_budget=season_budget.sort_values(ascending=False)
sns.barplot(y=season_budget.values,x=season_budget.index)
plt.figure(figsize=(10,10))
sns.jointplot(x="Area (Hectare)", y="Production (Tonnes)", data=df,kind='reg')
plt.xticks(rotation=90)
Split the data
data1 = df.drop(["District","Year"],axis=1)
data2 = df.drop(["District"],axis=1)
data_dum = pd.get_dummies(data1)
data_dum1 = pd.get_dummies(data2)
data_dum[:5]
df.dtypes
x = data_dum.drop("Production (Tonnes)",axis=1)
y = data_dum[["Production (Tonnes)"]]
x1 = data_dum1.drop("Production (Tonnes)",axis=1)
y1 = data_dum1[["Production (Tonnes)"]]
x.shape
y.shape
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_dataS = scaler.fit_transform(x)
x = pd.DataFrame(scaled_dataS,
						columns=x.columns)
print(x.head())
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_dataS = scaler.fit_transform(x1)
x1 = pd.DataFrame(scaled_dataS,
						columns=x1.columns)
print(x1.head())
scaled_dataS = scaler.fit_transform(y)
y = pd.DataFrame(scaled_dataS,
						columns=y.columns)
print(y.head())
scaled_dataS = scaler.fit_transform(y1)
y1 = pd.DataFrame(scaled_dataS,
						columns=y1.columns)
print(y1.head())
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.4, random_state=42)
print("x_train :",x_train.shape)
print("x_test :",x_test.shape)
print("y_train :",y_train.shape)
print("y_test :",y_test.shape)
from sklearn.model_selection import train_test_split
x_train1,x_test1,y_train1,y_test1 = train_test_split(x1,y1,test_size=0.4, random_state=42)
print("x_train :",x_train1.shape)
print("x_test :",x_test1.shape)
print("y_train :",y_train1.shape)
print("y_test :",y_test1.shape)
x_train[:5]
Model Training
Random Forest
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor()
model_rf.fit(x_train, y_train.values.ravel())
preds = model_rf.predict(x_test)
from sklearn.metrics import mean_squared_error, r2_score
mse_rf = mean_squared_error(y_test, preds)
r2_rf = r2_score(y_test, preds)

print("Mean Squared Error:", mse_rf)
print("R-squared:", r2_rf)
Linear Regression
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(x_train,y_train)

pickle.dump(regressor,open('model.pkl','wb'))
model=pickle.load(open('model.pkl','rb'))
preds = model.predict(x_test)

from sklearn.metrics import mean_squared_error, r2_score
mse_lr = mean_squared_error(y_test, preds)
r2_lr = r2_score(y_test, preds)

print("Mean Squared Error:", mse_lr)
print("R-squared:", r2_lr)
Lasso Regression
import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.model_selection import RepeatedKFold

cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)

model_lasso = LassoCV(alphas=np.array([0.01]), cv=cv, n_jobs=-1)

model_lasso.fit(x_train, y_train.values.ravel())
preds = model_lasso.predict(x_test)

mse_lasso = mean_squared_error(y_test, preds)
r2_lasso = r2_score(y_test, preds)

print("Mean Squared Error:", mse_lasso)
print("R-squared:", r2_lasso)
Decision Regression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
model_dt = DecisionTreeRegressor(random_state=42)
model_dt.fit(x_train, y_train)

predictions = model_dt.predict(x_test)

mse_dt = mean_squared_error(y_test, predictions)
r2_dt = r2_score(y_test, predictions)

print("Mean Squared Error:", mse_dt)
print("R-squared:", r2_dt)

KNN
from sklearn.neighbors import KNeighborsRegressor
scores = []

for i in range(1,16):

    knn = KNeighborsRegressor(n_neighbors=i)

    knn.fit(x_train,y_train)

    y_pred = knn.predict(x_test)

    scores.append(r2_score(y_test, y_pred)
)
import matplotlib.pyplot as plt
plt.figure(figsize=(18,10))
plt.plot(range(1,16),scores)
model_knn = KNeighborsRegressor(n_neighbors=2)
model_knn.fit(x_train, y_train)

y_pred = model.predict(x_test)

mse_knn = mean_squared_error(y_test, y_pred)
r2_knn = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse_knn)
print("R-squared:", r2_knn)

XG Boost
from xgboost import XGBRegressor
model = XGBRegressor()
model.fit(x_train, y_train)

y_pred = model.predict(x_test)

mse_xg = mean_squared_error(y_test, y_pred)
r2_xg = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse_xg)
print("R-squared:", r2_xg)
Deep Neural Networks
Convolutional Neural Network
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
model = Sequential([
    Dense(64, activation='relu', input_shape=(45,)),
    Dense(64, activation='relu'),
    Dense(1)  # Assuming you have a regression task, change to Dense(num_classes, activation='softmax') for classification
])

# Compile the model
model.compile(optimizer='adam', loss='mse')  # Use 'categorical_crossentropy' for classification

# Train the model
model.fit(x_train, y_train, epochs=50, batch_size=32, validation_data=(x_test, y_test))
y_pred = model.predict(x_test)

mse_nn = mean_squared_error(y_test, y_pred)
r2_nn = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse_nn)
print("R-squared:", r2_nn)
Long Short termed Memory
from tensorflow.keras.layers import LSTM, Dense

# Assuming x_train and x_test are pandas DataFrames
x_train = x_train.values.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.values.reshape((x_test.shape[0], x_test.shape[1], 1))

# Create the LSTM model
model = Sequential([
    LSTM(64, input_shape=(x_train.shape[1], 1)),
    Dense(1)  # Assuming you have a regression task
])

# Compile the model
model.compile(optimizer='adam', loss='mse')  # Use 'categorical_crossentropy' for classification

# Train the model
model.fit(x_train, y_train, epochs=50, batch_size=32, validation_data=(x_test, y_test))
# Make predictions
y_pred = model.predict(x_test)

mse_lstm = mean_squared_error(y_test, y_pred)
r2_lstm = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse_lstm)
print("R-squared:", r2_lstm)


# R-squared values for each model
r2_values = {
    'Linear Regression': r2_lr,
    'KNN Regressor': r2_knn,
    'Decision Tree': r2_dt,
    'XGBoost': r2_xg,
    'Random Forest': r2_rf,
    'Neural Network': r2_nn,
    'LSTM': r2_lstm
}

# Plotting
plt.figure(figsize=(10, 6))
plt.bar(r2_values.keys(), r2_values.values(), color='skyblue')
plt.title('R-squared Values of Regression Models')
plt.xlabel('Model')
plt.ylabel('R-squared')
plt.ylim(0, 1)  # Set the y-axis limits to ensure visibility
plt.xticks(rotation=45)
plt.show()
import seaborn as sns

r2_df = pd.DataFrame(r2_values.items(), columns=['Model', 'R-squared'])

# Create a point plot
plt.figure(figsize=(10, 6))
sns.pointplot(data=r2_df, x='Model', y='R-squared', color='blue')
plt.title('R-squared Values of Regression Models')
plt.xlabel('Model')
plt.ylabel('R-squared')
plt.xticks(rotation=0)
plt.grid(True)
plt.show()
CNN-LSTM
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Flatten
x_train_array = x_train.to_numpy()
y_train_array = y_train.to_numpy()

# Reshape the input data for CNN (assuming the number of features is 1)
x_train_cnn = x_train_array.reshape((x_train_array.shape[0], x_train_array.shape[1], 1))


# Create the CNN-LSTM model
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(x_train.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    LSTM(50),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')  

# Fit the model
model.fit(x_train_cnn, y_train, epochs=50, batch_size=32, validation_split=0.2)
# Make predictions
y_pred = model.predict(x_test)

mse_cnnlstm = mean_squared_error(y_test, y_pred)
r2_cnnlstm = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse_cnnlstm)
print("R-squared:", r2_cnnlstm)

def recommend_crop(temp, rain):
    if rain > 200:
        return "Rice"
    elif temp > 30:
        return "Maize"
    else:
        return "Wheat"

# response
return jsonify({
    "prediction": round(float(pred), 2),
    "temp": temp,
    "humidity": hum,
    "rainfall": rain,
    "ai_crop": recommend_crop(temp, rain)
})
from flask import Flask, render_template, request, jsonify
from flask_caching import Cache
import logging

app = Flask(__name__)
cache = Cache(app, config={'CACHE_TYPE': 'simple'})

# Logging setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@cache.memoize(timeout=3600)  # Cache weather for 1 hour
def get_weather_cached(city):
    return get_weather(city)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json() or request.form
        
        state = data.get('state')
        district = data.get('district')
        season = data.get('season')
        year = int(data.get('year'))
        area = float(data.get('area'))
        
        # Get weather data
        temp, humidity, rain = get_weather_cached(district)
        month = int(data.get('month', 6))
        
        # Prepare input
        input_df = prepare_input(state, district, season, year, area, temp, humidity, rain)
        scaled_input = scaler.transform(input_df)
        
        # Get predictions from multiple models
        predictions = {
            'xgboost': float(xgb_model.predict(scaled_input)[0]),
            'random_forest': float(rf_model.predict(scaled_input)[0]),
            'stacking': float(stacking_model.predict(scaled_input)[0])
        }
        
        # Ensemble prediction (weighted average)
        weights = {'xgboost': 0.4, 'random_forest': 0.3, 'stacking': 0.3}
        final_prediction = sum(pred * weights[name] for name, pred in predictions.items())
        
        # Get crop recommendation
        crop_rec = recommend_crop_advanced(temp, rain, humidity, month=month)
        
        # Confidence interval (using prediction std across models)
        pred_values = list(predictions.values())
        confidence_interval = {
            'lower': min(pred_values) * 0.95,
            'upper': max(pred_values) * 1.05
        }
        
        return jsonify({
            'status': 'success',
            'prediction': round(final_prediction, 2),
            'prediction_tonnes': round(scaler.inverse_transform([[final_prediction]])[0][0], 2),
            'confidence_interval': confidence_interval,
            'model_predictions': predictions,
            'weather': {
                'temperature': temp,
                'humidity': humidity,
                'rainfall': rain
            },
            'crop_recommendation': crop_rec
        })
        
    except Exception as e:
        logger.error(f"Prediction error: {str(e)}")
        return jsonify({'status': 'error', 'message': str(e)}), 400

@app.route('/analyze/trends', methods=['GET'])
def analyze_trends():
    """Historical trend analysis endpoint"""
    state = request.args.get('state')
    
    state_data = df[df['State'] == state].groupby('Year').agg({
        'Production (Tonnes)': ['mean', 'sum', 'std'],
        'Area (Hectare)': 'sum',
        'Yield (Tonnes/Hectare)': 'mean'
    }).reset_index()
    
    # Calculate year-over-year growth
    state_data['YoY_Growth'] = state_data[('Production (Tonnes)', 'sum')].pct_change() * 100
    
    return jsonify({
        'state': state,
        'trends': state_data.to_dict(orient='records'),
        'summary': {
            'avg_production': float(state_data[('Production (Tonnes)', 'mean')].mean()),
            'avg_growth': float(state_data['YoY_Growth'].mean()),
            'peak_year': int(state_data.loc[
                state_data[('Production (Tonnes)', 'sum')].idxmax(), 'Year'
            ])
        }
    })

if __name__ == '__main__':
    app.run(debug=True, threaded=True)



from flask import Flask, render_template, request, jsonify
from flask_caching import Cache
import logging

app = Flask(__name__)
cache = Cache(app, config={'CACHE_TYPE': 'simple'})

# Logging setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@cache.memoize(timeout=3600)  # Cache weather for 1 hour
def get_weather_cached(city):
    return get_weather(city)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json() or request.form
        
        state = data.get('state')
        district = data.get('district')
        season = data.get('season')
        year = int(data.get('year'))
        area = float(data.get('area'))
        
        # Get weather data
        temp, humidity, rain = get_weather_cached(district)
        month = int(data.get('month', 6))
        
        # Prepare input
        input_df = prepare_input(state, district, season, year, area, temp, humidity, rain)
        scaled_input = scaler.transform(input_df)
        
        # Get predictions from multiple models
        predictions = {
            'xgboost': float(xgb_model.predict(scaled_input)[0]),
            'random_forest': float(rf_model.predict(scaled_input)[0]),
            'stacking': float(stacking_model.predict(scaled_input)[0])
        }
        
        # Ensemble prediction (weighted average)
        weights = {'xgboost': 0.4, 'random_forest': 0.3, 'stacking': 0.3}
        final_prediction = sum(pred * weights[name] for name, pred in predictions.items())
        
        # Get crop recommendation
        crop_rec = recommend_crop_advanced(temp, rain, humidity, month=month)
        
        # Confidence interval (using prediction std across models)
        pred_values = list(predictions.values())
        confidence_interval = {
            'lower': min(pred_values) * 0.95,
            'upper': max(pred_values) * 1.05
        }
        
        return jsonify({
            'status': 'success',
            'prediction': round(final_prediction, 2),
            'prediction_tonnes': round(scaler.inverse_transform([[final_prediction]])[0][0], 2),
            'confidence_interval': confidence_interval,
            'model_predictions': predictions,
            'weather': {
                'temperature': temp,
                'humidity': humidity,
                'rainfall': rain
            },
            'crop_recommendation': crop_rec
        })
        
    except Exception as e:
        logger.error(f"Prediction error: {str(e)}")
        return jsonify({'status': 'error', 'message': str(e)}), 400

@app.route('/analyze/trends', methods=['GET'])
def analyze_trends():
    """Historical trend analysis endpoint"""
    state = request.args.get('state')
    
    state_data = df[df['State'] == state].groupby('Year').agg({
        'Production (Tonnes)': ['mean', 'sum', 'std'],
        'Area (Hectare)': 'sum',
        'Yield (Tonnes/Hectare)': 'mean'
    }).reset_index()
    
    # Calculate year-over-year growth
    state_data['YoY_Growth'] = state_data[('Production (Tonnes)', 'sum')].pct_change() * 100
    
    return jsonify({
        'state': state,
        'trends': state_data.to_dict(orient='records'),
        'summary': {
            'avg_production': float(state_data[('Production (Tonnes)', 'mean')].mean()),
            'avg_growth': float(state_data['YoY_Growth'].mean()),
            'peak_year': int(state_data.loc[
                state_data[('Production (Tonnes)', 'sum')].idxmax(), 'Year'
            ])
        }
    })

if __name__ == '__main__':
    app.run(debug=True, threaded=True)

def recommend_crop_advanced(temp, rain, humidity, soil_type=None, month=None):
    """
    Multi-factor crop recommendation using rule-based + ML hybrid approach
    """
    recommendations = []
    confidence_scores = {}
    
    # Rice conditions
    rice_score = 0
    if 20 <= temp <= 35:
        rice_score += 30
    if rain > 150:
        rice_score += 35
    if humidity > 60:
        rice_score += 20
    if month in [6, 7, 8]:  # Kharif season
        rice_score += 15
    confidence_scores['Rice'] = rice_score
    
    # Wheat conditions
    wheat_score = 0
    if 10 <= temp <= 25:
        wheat_score += 35
    if 50 <= rain <= 100:
        wheat_score += 30
    if humidity < 70:
        wheat_score += 20
    if month in [10, 11, 12]:  # Rabi season
        wheat_score += 15
    confidence_scores['Wheat'] = wheat_score
    
    # Maize conditions
    maize_score = 0
    if 21 <= temp <= 30:
        maize_score += 30
    if 50 <= rain <= 150:
        maize_score += 30
    if humidity > 50:
        maize_score += 20
    confidence_scores['Maize'] = maize_score
    
    # Sugarcane conditions
    sugarcane_score = 0
    if 25 <= temp <= 35:
        sugarcane_score += 35
    if rain > 100:
        sugarcane_score += 30
    if humidity > 70:
        sugarcane_score += 20
    confidence_scores['Sugarcane'] = sugarcane_score
    
    # Sort by confidence
    sorted_crops = sorted(confidence_scores.items(), key=lambda x: x[1], reverse=True)
    
    return {
        'primary_recommendation': sorted_crops[0][0],
        'confidence': sorted_crops[0][1],
        'alternatives': [{'crop': c, 'score': s} for c, s in sorted_crops[1:3]],
        'all_scores': confidence_scores
    }

import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10)
    }
    
    model = XGBRegressor(**params, random_state=42)
    scores = cross_val_score(model, x_train, y_train.values.ravel(), 
                             cv=5, scoring='r2', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"Best R²: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

# Train final model with best params
best_xgb = XGBRegressor(**study.best_params, random_state=42)
best_xgb.fit(x_train, y_train.values.ravel())

df = pd.read_excel('Data/APY_Rice.xls')
# =========================
# IMPORTS (same)
# =========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline


# =========================
# LOAD DATA (FIXED PATH)
# =========================
df = pd.read_excel('Data/APY_Rice.xls')   # <-- only change

print(df.head())   # <-- fixed (brackets added)


# =========================
# BASIC INFO (same)
# =========================
print(df.info())
print(df.describe())


# =========================
# MISSING VALUES (same)
# =========================
print(df.isnull().sum())

df.dropna(inplace=True)


# =========================
# OUTLIER REMOVAL (FIXED ONLY LOGIC)
# =========================
cols = df.select_dtypes(include=np.number).columns

for temp in cols:
    Q1 = df[temp].quantile(0.25)
    Q3 = df[temp].quantile(0.75)
    IQR = Q3 - Q1

    threshold = 1.5
    
    lower = Q1 - threshold * IQR
    upper = Q3 + threshold * IQR

    df = df[(df[temp] >= lower) & (df[temp] <= upper)]   # <-- fixed


# =========================
# VISUALIZATION (same but fixed loop error)
# =========================
for col in cols:
    plt.figure(figsize=(7,7))
    sns.boxplot(x=df[col])
    plt.title(col)
    plt.show()


# =========================
# CORRELATION (same)
# =========================
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.show()


# =========================
# MODEL PART (minimal fix)
# =========================
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)


# =========================
# PREDICTION (same flow)
# =========================
y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))


# =========================
# SAMPLE OUTPUT (added minimal)
# =========================
sample = X_test.iloc[0].values.reshape(1, -1)
print("Sample Prediction:", model.predict(sample))